# CT.M1 - Data Acquisition
### Notebook 02. PUG-REST and HTTP-POST Requests 

**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/02_pugrest_post_requests.ipynb)


---


### Contents

This notebook advances your data acquisition skills by exploring **HTTP POST** requests, a method used when simple text searches are not enough. You will learn how to communicate with `PubChem` not just by "naming" a molecule, but by "showing" its actual chemical architecture.

The activities are structured to guide you through the process of:

1. Structure-Based Querying: Learn how to use a chemical representation (SMILES or an SDF file) as the starting point for a search, rather than just a name or CID.

2. GET vs. POST Logic: Understand the technical difference between sending data in the "Header" (the URL) versus the "Body" (the Payload), and why the latter is essential for complex structural data.

3. Mastering requests.post(): Implement Python code to upload molecular files and handle server responses for "Virtual Screening" tasks.

---


### How to work

1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourName_YourID_02_pugrest_post_requests.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

## The HTTP POST Method

**PUG-REST** (Power User Gateway-Representational State Transfer) handles short requests where results are provided in a single call with a maximum duration of 30 seconds. This eliminates the need for intermediate steps to check if the request is complete. PUG-REST requests can be executed using either **HTTP-GET** or **HTTP-POST** methods. These methods offer different levels of cybersecurity and varying degrees of convenience, depending on the specific requirements of the research task.

In our previous notebook, we asked `PubChem` for data using a simple URL `(HTTP GET)`. This is like calling a library and asking for a book by its title. 

But what if you have a **map** of a new molecule you designed? Or a complex file containing rare characters (**InChIKey**) or 3D coordinates (**SDF**)? 

These "packages" are too big to fit in a URL. To handle this, we use the `HTTP POST` method. 

**Think of it this way:**
* **HTTP GET:** Sending a postcard (everyone sees it, it must be short).
* **HTTP POST:** Sending a sealed parcel (it can carry heavy files and complex instructions).


---

## GET vs. POST: Choosing the Right Tool for the Task

Table 1 analyzes the key differences between these two methods. Understanding these distinctions is crucial for deciding how to structure your requests to PubChem depending on the complexity and sensitivity of your research data.

### Table 1. Differences between HTTP GET and POST methods

| Feature | HTTP GET | HTTP POST |
| :--- | :--- | :--- |
| **Data Transmission** | Sends variables encoded directly in the URL to execute the request. | Sends variables as part of the data "payload"; therefore, they are not visible in the URL. |
| **Security** | **Less secure:** Data is visible in the URL and is stored in browser history and web server logs. | **More secure:** Data is not stored in browser history or server logs, making it harder to intercept. |
| **Size Limit** | Average limit of **2048 characters** for the entire URL. | **No limit** for the size of the data being sent. |
| **Character Types** | Supports only **ASCII characters** (0-9, English alphabet, and basic symbols). | No restrictions on character types; even supports **binary data**. |
| **Primary Use** | Exclusively used to **request** or retrieve data from servers. | Used to **send** complex data to servers and can be used to trigger server-side modifications. |

In the real world, a researcher doesn't always look for `aspirin`. Sometimes, they sketch a molecule in a software, save it as an `.SDF` file, and need to find out: **Does this exist in PubChem, and what are its properties?** 

This notebook teaches you how to answer that question.

---


### Step 0: Setting the Gateway

Initialize the working environment and define the "address" of the PubChem server. This establishes the foundation for all subsequent programmatic calls to the **PUG-REST** service.

In the code cell below, ensure that the `BASE_URL` is correctly pointed to the PubChem PUG-REST entry point. This variable will be the prefix for every request you build in this notebook.

No visible output is expected yet, but the variables will be stored in your computer's memory. If you run the cell and no error appears, you are ready to start communicating with PubChem.

In [ ]:
import requests
import pandas as pd
import time
from io import StringIO

# TODO: Define the PUG-REST base URL
BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

print("Setup complete.")

### Step 1: GET vs. POST Communication

Before writing code, we must distinguish between the two main "languages" we use to talk to PubChem. While they both retrieve data, they handle the information load differently.

**Concept**:

- **HTTP GET** (The Postcard): You put all your instructions directly into the URL (the address bar). It is simple and fast, but it has a character limit (around 2,048 characters). It is perfect for queries like "Give me the CID for `aspirin`".

- **HTTP POST** (The Parcel): You send the data inside the Request Body (Payload). The URL stays short and clean, while the "heavy" data is hidden inside the message. It is essential for sending complex chemical structures (SMILES/SDF) or massive lists of thousands of compounds.


    >**Note**: If you ever see a **414 Request-URI Too Large** error, it means your "postcard" (GET) was too small for your data. That is the signal to switch to a "parcel" (POST).

In [ ]:
# Step 1

query_name = "aspirin"

# TODO: build the GET URL to retrieve CIDs from a name
url = f"{BASE_URL}/compound/name/{query_name}/cids/JSON"

r = requests.get(url)
r.raise_for_status()

data = r.json()

cid = data["IdentifierList"]["CID"][0]
print("CID for aspirin:", cid)

### Step 2: Structure-Based Query Using POST

When we send a structure to PubChem, we are usually performing an **Identity Search**. This is like a fingerprint scan: PubChem looks for a molecule that matches your input exactly in terms of connectivity and atoms.

#### Types of Structural Input:
1.  SMILES (Line Notation): A way of writing a 2D molecule as a single string of text.
    > Example: `CC(=O)OC1=CC=CC=C1C(=O)O` (aspirin).

2. SDF / Molfile (The Map): A multi-line file format that includes the 3D coordinates ($x, y, z$) of every atom. It looks like a complex table of numbers and is too large to fit in a URL.


#### Identity vs. Similarity (A Sneak Peek):
- Identity Search: "Find me the exact same molecule."
- Similarity Search: "Find me molecules that look like or have a similar skeleton to this one." (We will explore this in later modules).

In the upcoming code cell, you will define a chemical structure as a string and configure a `POST` request to ask PubChem: *"Does this structure exist in your database?* *If so, give me its CID.*"

In [ ]:
# Step 2

smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin

# TODO: choose an endpoint for structure identity search
url = f"{BASE_URL}/compound/smiles/cids/JSON"

# TODO: build payload for POST
payload = {"smiles": smiles}

r = requests.post(url, data=payload)
r.raise_for_status()

data = r.json()

print("Returned CIDs:", data["IdentifierList"]["CID"][:5])

### Step 3: Troubleshooting & Error Handling (When POST Fails)

In the real world, APIs don't always return a "200 OK". You might send a malformed SMILES, exceed the 30-second time limit, or request a resource that doesn't exist. This step teaches you how to "listen" to what the PubChem server is trying to tell you when things go wrong.

**Concept:**
When a requests.post() call fails, we need to inspect three key things:

1. HTTP Status Codes: * `400 Bad Request`: You likely have a typo in your URL or Payload.
    - `404 Not Found`: The structure or CID you provided is not in the database.
    - `414 URI Too Long`: You used GET when you should have used POST.
    - `504 Gateway Timeout`: Your request took longer than 30 seconds.

2. `r.text` **vs** `r.json()`: If the server returns an error, it often sends a text message explaining why. If you try to run `.json()` on an error message, your code will crash. Always check the status first!

3. **Timeouts**: Programmatic requests can "hang" forever if the server is busy. We use the timeout parameter to prevent our notebook from freezing.

In the code cell below, you will deliberately trigger a request and practice using `r.status_code` and `r.text` to debug the response. You will also implement a timeout to make your code more robust. Instead of a generic Python error, you should see a clear printout of the Status Code and the Server's explanation of the problem.

In [ ]:
# Step 3

def safe_post(url, payload, timeout=60):
    try:
        r = requests.post(url, data=payload, timeout=timeout)
        r.raise_for_status()
        return r
    except requests.exceptions.HTTPError:
        print("HTTPError:", r.status_code)
        print(r.text[:500])
        raise

print("Safe POST function defined.")

In [ ]:
# Take first 5 CIDs
cid_list = data["IdentifierList"]["CID"][:5]
cid_str = ",".join(map(str, cid_list))

props = [
    "SMILES",
    "InChIKey",
    "MolecularWeight",
    "XLogP",
    "TPSA",
    ]

# TODO: Build property retrieval URL
url_props = f"{BASE_URL}/compound/cid/{cid_str}/property/{','.join(props)}/JSON"

r2 = requests.get(url_props)
r2.raise_for_status()

props_data = r2.json()["PropertyTable"]["Properties"]

df = pd.DataFrame(props_data)
df

>**Note:** The number CID is implicit in the endpoint

When using:
    /compound/cid/{cid_list}/property/...

the CID is already defined in the URL.

Example:
    /compound/cid/2244/property/MolecularWeight,SMILES/JSON

Because of this:
- Do **not** include `"CID"` in the `props` list.
- PubChem automatically returns the CID column.
- Adding `"CID"` may cause a `400 BadRequest`.

### Step 4: Uploading an SDF File

Now we simulate a real research scenario.
Instead of SMILES, we upload a molecular file (`.SDF`).

This requires:
- POST request
- Multipart/form-data
- File upload handling

In [ ]:
# Step 4 

prolog = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
url = f"{prolog}/compound/sdf/cids/JSON"

sdf_text = """Aspirin
  PubChem

 21 21  0     0  0  0  0  0  0999 V2000
    3.7321   -0.0600    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301    1.4400    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    4.5981    1.4400    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    2.8660   -1.5600    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    4.5981   -0.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641   -0.0600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    4.5981   -1.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301   -0.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641   -2.0600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301   -1.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641    0.9400    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.8660   -0.5600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.0000   -0.0600    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    4.0611   -1.8700    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    6.8671   -0.2500    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    5.4641   -2.6800    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    6.8671   -1.8700    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    2.3100    0.4769    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.4631    0.2500    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.6900   -0.5969    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    6.3301    2.0600    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
  1  5  1  0  0  0  0
  1 12  1  0  0  0  0
  2 11  1  0  0  0  0
  2 21  1  0  0  0  0
  3 11  2  0  0  0  0
  4 12  2  0  0  0  0
  5  6  1  0  0  0  0
  5  7  2  0  0  0  0
  6  8  2  0  0  0  0
  6 11  1  0  0  0  0
  7  9  1  0  0  0  0
  7 14  1  0  0  0  0
  8 10  1  0  0  0  0
  8 15  1  0  0  0  0
  9 10  2  0  0  0  0
  9 16  1  0  0  0  0
 10 17  1  0  0  0  0
 12 13  1  0  0  0  0
 13 18  1  0  0  0  0
 13 19  1  0  0  0  0
 13 20  1  0  0  0  0
M  END
$$$$
"""


sdf_path = "query_molecule.sdf"
with open(sdf_path, "w", encoding="utf-8") as f:
    f.write(sdf_text)

print("Temporary SDF file created.")


In [ ]:
cid_list = data["IdentifierList"]["CID"]
cid_str = ",".join(map(str, cid_list))

props = [
    "SMILES",
    "InChIKey",
    "MolecularWeight",
    "XLogP",
    "TPSA"
]

url_props = f"{BASE_URL}/compound/cid/{cid_str}/property/{','.join(props)}/JSON"

r2 = requests.get(url_props)
print("Status:", r2.status_code)
print(r2.text[:300])

r2.raise_for_status()

props_data = r2.json()["PropertyTable"]["Properties"]

df = pd.DataFrame(props_data)
df

### Step 5: Virtual Screening Concept (ListKey)

Sometimes PubChem cannot return results immediately (e.g., large similarity searches).
Instead of CIDs, the server returns a temporary **ListKey** (a job ID).

Workflow:
1) Submit POST → receive **ListKey**
2) Poll using ListKey → wait until results are ready
3) Retrieve final CIDs using ListKey

In [ ]:
# Step 5.1 — Submit a "heavy" search (expects ListKey)

query_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin
threshold = 90  # lower threshold -> more hits -> more likely ListKey

url = f"{BASE_URL}/compound/smiles/fastsimilarity_2d/cids/JSON"
payload = {"smiles": query_smiles, "Threshold": threshold}

r = requests.post(url, data=payload, timeout=60)
print("Status:", r.status_code)
print(r.text[:400])   # show response preview to students
r.raise_for_status()

job = r.json()

# If PubChem returns ListKey, it will be here:
listkey = job["IdentifierList"]["ListKey"]
print("ListKey:", listkey)

In [ ]:
# Step 5.2 — Poll using ListKey until CIDs are ready

def poll_listkey(listkey, sleep=2, max_tries=30):
    url = f"{BASE_URL}/compound/listkey/{listkey}/cids/JSON"
    for i in range(max_tries):
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            return r.json()
        time.sleep(sleep)
    raise TimeoutError("Polling timed out.")

result = poll_listkey(listkey)
cids = result["IdentifierList"]["CID"]

print("Number of hits:", len(cids))
print("First 10 CIDs:", cids[:10])